# Практическое занятие №5
## Контролируемое обучение (Supervised Learning)

Этот ноутбук предназначен для выполнения ПЗ‑5 по лекции 3.1.

**Важно:** каждый студент выполняет работу по *своему варианту* (CSV‑файлу). Предполагается, что ноутбук и файлы данных находятся **в одной папке**.

## Как работать с ноутбуком

1. В ячейке **«Номер варианта»** укажите `VARIANT` от 1 до 15.
2. Запустите ноутбук сверху вниз.
3. В текстовых ячейках (где написано *«Ответ студента»*) впишите свои пояснения.

В конце у вас должен получиться отчёт: постановка задачи, описание данных, обучение модели, метрики, анализ переобучения и выводы.

In [ ]:
# Ячейка 1. Импорт библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    mean_squared_error, mean_absolute_error
)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
print('Библиотеки успешно импортированы.')

In [ ]:
# Ячейка 2. Номер варианта и загрузка данных
# УКАЖИТЕ НОМЕР СВОЕГО ВАРИАНТА (1–15)
VARIANT = 1

# Имя файла по шаблону: dataset_variant_1.csv, dataset_variant_2.csv, ...
filename = f"dataset_variant_{VARIANT}.csv"
print('Открываю файл:', filename)

df = pd.read_csv(filename)
display(df.head())
print('\nРазмер таблицы:', df.shape)

## 1. Первичный анализ данных (обязательный)

На этом шаге вы должны понять:
- какие столбцы есть в данных;
- какие типы данных у столбцов;
- есть ли пропуски;
- есть ли очевидные аномалии.


In [ ]:
# Ячейка 3. Информация о данных
df.info()
print('\nЧисло пропусков по столбцам:')
display(df.isna().sum())

print('\nОписательная статистика по числовым столбцам:')
display(df.describe(include=[np.number]))

### Ответ студента (кратко, 6–10 предложений)
1) Что это за данные (по смыслу)?
2) Сколько объектов и сколько признаков?
3) Есть ли пропуски? Если да — где и сколько?
4) Какие столбцы выглядят как кандидаты на целевую переменную? Почему?

## 2. Постановка задачи контролируемого обучения

Контролируемое обучение (*Supervised Learning*) означает, что для каждого объекта известен **правильный ответ**.

Вам нужно:
- определить, что у вас: **классификация** (цель — класс) или **регрессия** (цель — число);
- выбрать **целевой столбец** (target);
- обосновать выбор.

In [ ]:
# Ячейка 4. Выбор целевой переменной и признаков
# УКАЖИТЕ ИМЯ ЦЕЛЕВОГО СТОЛБЦА (как в вашем CSV)
TARGET_COLUMN = 'target'

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print('Целевая переменная:', TARGET_COLUMN)
print('Число признаков:', X.shape[1])
print('\nСписок признаков:')
display(pd.Series(X.columns))

print('\nТип целевой переменной:', y.dtype)
display(y.head())

### Ответ студента
1) Какой столбец вы выбрали в качестве целевой переменной и почему?
2) Это классификация или регрессия? Обоснуйте.
3) Какие 2–3 признака, по вашему мнению, наиболее важны для предсказания цели? Почему?

## 3. Разделение данных на обучающую и тестовую выборки

Мы должны проверить способность модели **обобщать** на новые данные.

Данные делят на:
- **обучающую выборку (train)** — на ней модель учится;
- **тестовую выборку (test)** — на ней оцениваем качество на данных, которые модель не видела.


In [ ]:
# Ячейка 5. Train/Test split
TEST_SIZE = 0.30
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print('Размер train:', X_train.shape)
print('Размер test :', X_test.shape)

## 4. Выбор модели и обучение

В учебной версии ПЗ‑5 используем:
- для **классификации** — `LogisticRegression` (логистическая регрессия);
- для **регрессии** — `LinearRegression` (линейная регрессия).

**Важно:** тип задачи вы задаёте явно в `MODEL_TYPE`.

In [ ]:
# Ячейка 6. Обучение модели
MODEL_TYPE = 'classification'  # 'classification' или 'regression'

if MODEL_TYPE == 'classification':
    model = LogisticRegression(max_iter=2000)
elif MODEL_TYPE == 'regression':
    model = LinearRegression()
else:
    raise ValueError('MODEL_TYPE должен быть classification или regression')

model.fit(X_train, y_train)
print('Модель обучена:', type(model).__name__)

## 5. Прогнозирование и оценка качества

Считаем прогнозы на train и test и оцениваем метрики.

### Метрики
- Классификация: **accuracy** и **confusion matrix**.
- Регрессия: **MSE**, **RMSE**, **MAE**.


In [ ]:
# Ячейка 7. Прогнозирование
y_train_pred = model.predict(X_train)
y_test_pred  = model.predict(X_test)
print('Прогнозы рассчитаны.')

In [ ]:
# Ячейка 8. Метрики качества
if MODEL_TYPE == 'classification':
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc  = accuracy_score(y_test,  y_test_pred)
    print(f'Accuracy (train): {train_acc:.4f}')
    print(f'Accuracy (test) : {test_acc:.4f}')
    
    cm = confusion_matrix(y_test, y_test_pred)
    print('\nConfusion matrix (test):')
    display(pd.DataFrame(cm))
    
    plt.figure()
    plt.imshow(cm)
    plt.title('Матрица несоответствий (test)')
    plt.xlabel('Предсказанный класс')
    plt.ylabel('Истинный класс')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha='center', va='center')
    plt.tight_layout()
    plt.show()
else:
    mse  = mean_squared_error(y_test, y_test_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_test, y_test_pred)
    print(f'MSE : {mse:.4f}')
    print(f'RMSE: {rmse:.4f}')
    print(f'MAE : {mae:.4f}')
    
    plt.figure()
    plt.scatter(y_test, y_test_pred)
    plt.title('Истинные значения vs Предсказания (test)')
    plt.xlabel('Истинные значения')
    plt.ylabel('Предсказания')
    plt.tight_layout()
    plt.show()

## 6. Анализ переобучения (overfitting) и обобщения (generalization)

Сравните качество на обучающей и тестовой выборках.

- Если на train качество существенно лучше, чем на test, это может быть признаком **переобучения**.
- Если качество низкое и на train, и на test — возможное **недообучение**.


### Ответ студента
1) Есть ли признаки переобучения? (Да/Нет)
2) На чём основан вывод? (сравнение метрик train/test)
3) Какие 2–3 улучшения можно предложить?

## 7. Итоговые выводы

Сформулируйте выводы:
1) Какая задача решалась?
2) Какие признаки и цель использовались?
3) Какое качество получилось и что оно означает?
4) Какие ограничения есть у модели?


### Ответ студента
- **Задача:** ...
- **Целевая переменная:** ...
- **Модель:** ...
- **Метрики:** ...
- **Вывод:** ...